# Network Intrusion Detection Pipeline (NSL-KDD)

This notebook covers the complete machine learning pipeline for network anomaly and attack classification, including:
1. **Data Downloading & Setup**: Fetching NSL-KDD training/testing sets.
2. **Data Preprocessing & Scaling**: Loading raw NSL-KDD datasets, mapping attack names to broad categories, and encoding categorical fields.
3. **Supervised Random Forest Models**: Training a binary classifier (normal vs. attack) and a multi-class classifier (attack category groups like DoS, Probe, R2L, U2R).
4. **Unsupervised PyTorch Autoencoder**: Training an anomaly detection model on normal traffic only to catch unseen attack categories.

## 1. Environment Setup & Data Fetching

In [ ]:
import os
import sys
import time
import urllib.request
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Set up directories relative to the notebook path
NOTEBOOK_DIR = Path(os.getcwd())
ROOT_DIR = NOTEBOOK_DIR.parent
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
MODELS_DIR = ROOT_DIR / "data" / "models"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {ROOT_DIR}")
print(f"Raw data dir: {RAW_DIR}")
print(f"Processed data dir: {PROCESSED_DIR}")
print(f"Models dir: {MODELS_DIR}")

In [ ]:
# Download the NSL-KDD dataset if not present
TRAIN_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt"
TEST_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"

train_path = RAW_DIR / "KDDTrain.txt"
test_path = RAW_DIR / "KDDTest.txt"

if not train_path.exists():
    print("Downloading KDDTrain.txt...")
    urllib.request.urlretrieve(TRAIN_URL, train_path)
else:
    print("KDDTrain.txt already exists locally.")

if not test_path.exists():
    print("Downloading KDDTest.txt...")
    urllib.request.urlretrieve(TEST_URL, test_path)
else:
    print("KDDTest.txt already exists locally.")

## 2. Preprocessing & Data Schema Setup

We define the schema and map the specific attack types into broad categories:
- **DoS** (Denial of Service)
- **Probe** (Network surveillance/scanning)
- **R2L** (Remote-to-Local user unauthorized access)
- **U2R** (User-to-Root privilege escalation)

In [ ]:
NSL_KDD_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty",
]

CATEGORICAL_COLUMNS = ["protocol_type", "service", "flag"]

ATTACK_CATEGORY_MAP = {
    # DoS
    "back": "dos", "land": "dos", "neptune": "dos", "pod": "dos", "smurf": "dos",
    "teardrop": "dos", "apache2": "dos", "udpstorm": "dos", "processtable": "dos",
    "worm": "dos", "mailbomb": "dos",
    # Probe
    "satan": "probe", "ipsweep": "probe", "nmap": "probe", "portsweep": "probe",
    "mscan": "probe", "saint": "probe",
    # R2L
    "guess_passwd": "r2l", "ftp_write": "r2l", "imap": "r2l", "phf": "r2l",
    "multihop": "r2l", "warezmaster": "r2l", "warezclient": "r2l", "spy": "r2l",
    "xlock": "r2l", "xsnoop": "r2l", "snmpguess": "r2l", "snmpgetattack": "r2l",
    "httptunnel": "r2l", "sendmail": "r2l", "named": "r2l",
    # U2R
    "buffer_overflow": "u2r", "loadmodule": "u2r", "rootkit": "u2r", "perl": "u2r",
    "sqlattack": "u2r", "xterm": "u2r", "ps": "u2r",
}

class SafeLabelEncoder:
    def __init__(self):
        self.le = LabelEncoder()
        self.classes_ = None

    def fit(self, values):
        values = list(values) + ["UNK"]
        self.le.fit(values)
        self.classes_ = self.le.classes_
        return self

    def transform(self, values):
        known = set(self.classes_)
        safe_values = [v if v in known else "UNK" for v in values]
        return self.le.transform(safe_values)

def add_targets(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["binary_label"] = (df["label"] != "normal").astype(int)
    df["category_label"] = df["label"].map(
        lambda x: "normal" if x == "normal" else ATTACK_CATEGORY_MAP.get(x, "other")
    )
    return df

In [ ]:
print("Loading raw datasets...")
train_df = pd.read_csv(train_path, header=None, names=NSL_KDD_COLUMNS)
test_df = pd.read_csv(test_path, header=None, names=NSL_KDD_COLUMNS)

train_df = add_targets(train_df)
test_df = add_targets(test_df)

print(f"Train size: {train_df.shape}, Test size: {test_df.shape}")

In [ ]:
# Visualize Label Distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.countplot(x="binary_label", data=train_df, palette="viridis")
plt.title("Binary Target Distribution (Train)")
plt.xticks([0, 1], ["Normal (0)", "Attack (1)"])

plt.subplot(1, 2, 2)
sns.countplot(y="category_label", data=train_df, order=train_df["category_label"].value_counts().index, palette="rocket")
plt.title("Attack Category Distribution (Train)")

plt.tight_layout()
plt.show()

In [ ]:
# Encode Categorical Variables safely
encoders = {}
for col in CATEGORICAL_COLUMNS:
    enc = SafeLabelEncoder()
    enc.fit(train_df[col])
    train_df[col + "_enc"] = enc.transform(train_df[col])
    test_df[col + "_enc"] = enc.transform(test_df[col])
    encoders[col] = enc
    print(f"Encoded {col}: fit {len(enc.classes_)} classes.")

# Select feature columns
feature_cols = [c for c in NSL_KDD_COLUMNS if c not in ("label",) + tuple(CATEGORICAL_COLUMNS)]
feature_cols = [c for c in feature_cols if c != "difficulty"]
feature_cols += [c + "_enc" for c in CATEGORICAL_COLUMNS]

X_train = train_df[feature_cols].astype(float).values
X_test = test_df[feature_cols].astype(float).values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_binary = train_df["binary_label"].values
y_test_binary = test_df["binary_label"].values

cat_encoder = LabelEncoder()
y_train_cat = cat_encoder.fit_transform(train_df["category_label"])
known_cats = set(cat_encoder.classes_)
test_cats_safe = test_df["category_label"].map(lambda c: c if c in known_cats else "other")
if "other" not in known_cats:
    cat_encoder.classes_ = np.append(cat_encoder.classes_, "other")
y_test_cat = cat_encoder.transform(test_cats_safe)

# Save processed arrays and transformers
np.save(PROCESSED_DIR / "X_train.npy", X_train_scaled)
np.save(PROCESSED_DIR / "X_test.npy", X_test_scaled)
np.save(PROCESSED_DIR / "y_train_binary.npy", y_train_binary)
np.save(PROCESSED_DIR / "y_test_binary.npy", y_test_binary)
np.save(PROCESSED_DIR / "y_train_cat.npy", y_train_cat)
np.save(PROCESSED_DIR / "y_test_cat.npy", y_test_cat)

joblib.dump(scaler, MODELS_DIR / "scaler.joblib")
joblib.dump(encoders, MODELS_DIR / "categorical_encoders.joblib")
joblib.dump(cat_encoder, MODELS_DIR / "category_label_encoder.joblib")
joblib.dump(feature_cols, MODELS_DIR / "feature_columns.joblib")

print(f"Preprocessing done. Scaled shape: {X_train_scaled.shape}")

## 3. Supervised Model: Random Forest

In [ ]:
print("=== Training RandomForest: BINARY (normal vs attack) ===")
rf_bin = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",
)
t0 = time.time()
rf_bin.fit(X_train_scaled, y_train_binary)
print(f"Binary RF trained in {time.time() - t0:.2f}s")

y_pred_bin = rf_bin.predict(X_test_scaled)
y_proba_bin = rf_bin.predict_proba(X_test_scaled)[:, 1]

print("\nClassification Report (Binary):")
print(classification_report(y_test_binary, y_pred_bin, target_names=["normal", "attack"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test_binary, y_pred_bin))
print(f"ROC-AUC: {roc_auc_score(y_test_binary, y_proba_bin):.4f}")

joblib.dump(rf_bin, MODELS_DIR / "rf_binary.joblib")

In [ ]:
print("=== Training RandomForest: MULTI-CLASS (attack category) ===")
rf_cat = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",
)
t0 = time.time()
rf_cat.fit(X_train_scaled, y_train_cat)
print(f"Multi-class RF trained in {time.time() - t0:.2f}s")

y_pred_cat = rf_cat.predict(X_test_scaled)
labels_present = sorted(set(y_test_cat) | set(y_pred_cat))
target_names = [cat_encoder.classes_[i] for i in labels_present]

print("\nClassification Report (Multi-Class):")
print(classification_report(y_test_cat, y_pred_cat, labels=labels_present, target_names=target_names))

joblib.dump(rf_cat, MODELS_DIR / "rf_category.joblib")

In [ ]:
# Visualizing Feature Importance
importances = rf_bin.feature_importances_
indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(10, 5))
sns.barplot(x=importances[indices], y=[feature_cols[i] for i in indices], palette="mako")
plt.title("Top 15 Feature Importances (Random Forest Binary)")
plt.xlabel("Relative Importance")
plt.ylabel("Features")
plt.show()

## 4. Unsupervised Model: PyTorch Autoencoder

An autoencoder trains *only* on normal traffic samples. Since it never sees attack payloads during training, it reconstructs attacks poorly. We evaluate the reconstruction error threshold (e.g. 95th percentile) to flag anomalies.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

class Autoencoder(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16), nn.ReLU(),
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
# Filter out only normal rows for training
X_normal_train = X_train_scaled[y_train_binary == 0]
input_dim = X_normal_train.shape[1]

model = Autoencoder(input_dim).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

X_tensor = torch.tensor(X_normal_train, dtype=torch.float32).to(DEVICE)
n_samples = X_tensor.shape[0]
batch_size = 256
epochs = 20

print(f"Training autoencoder on {n_samples} normal samples...")
model.train()
losses = []
for epoch in range(epochs):
    perm = torch.randperm(n_samples)
    epoch_loss = 0.0
    n_batches = 0
    for i in range(0, n_samples, batch_size):
        idx = perm[i : i + batch_size]
        batch = X_tensor[idx]
        optimizer.zero_grad()
        recon = model(batch)
        loss = loss_fn(recon, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch + 1}/{epochs} - Avg Loss: {avg_loss:.6f}")

In [ ]:
# Plot training loss curve
plt.figure(figsize=(6, 3))
plt.plot(losses, marker='o', color='purple')
plt.title("Autoencoder Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True)
plt.show()

In [ ]:
def compute_recon_errors(model, data):
    model.eval()
    with torch.no_grad():
        tensor = torch.tensor(data, dtype=torch.float32).to(DEVICE)
        recon = model(tensor)
        errors = torch.mean((recon - tensor) ** 2, dim=1)
    return errors.cpu().numpy()

train_errors_normal = compute_recon_errors(model, X_normal_train)
threshold = float(np.percentile(train_errors_normal, 95))
print(f"Reconstruction Error Threshold (95th percentile): {threshold:.6f}")

test_errors = compute_recon_errors(model, X_test_scaled)
y_pred_ae = (test_errors > threshold).astype(int)

print("\n=== Autoencoder Evaluation on Test Set ===")
print(classification_report(y_test_binary, y_pred_ae, target_names=["normal", "attack"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test_binary, y_pred_ae))
print(f"ROC-AUC: {roc_auc_score(y_test_binary, test_errors):.4f}")

In [ ]:
# Plot reconstruction error distributions
plt.figure(figsize=(10, 5))
sns.histplot(x=test_errors, hue=y_test_binary, kde=True, bins=50, palette="coolwarm", log_scale=True)
plt.axvline(x=threshold, color='red', linestyle='--', label=f'Threshold ({threshold:.4f})')
plt.title("Reconstruction Error Distribution (Test Set)")
plt.xlabel("MSE Reconstruction Error (Log Scale)")
plt.ylabel("Count")
plt.legend([f"Threshold ({threshold:.4f})", "Normal", "Attack"])
plt.show()

In [ ]:
# Save PyTorch model and metadata
torch.save(model.state_dict(), MODELS_DIR / "autoencoder.pt")
joblib.dump({"threshold": threshold, "input_dim": input_dim}, MODELS_DIR / "autoencoder_meta.joblib")
print("Autoencoder and metadata saved successfully.")